# Melampo: rilettura su CPU di un giro del probe (nessuna GPU)

Usa le regioni di VoxTell e le strutture di TotalSegmentator salvate dall'ultimo giro di `melampo-imaging-probe` (`--save-masks`), che Kaggle monta come dati in ingresso, e rifà solo l'analisi: misure, letture da radiologo, zone, margini. Serve quando cambia l'analisi ma non la segmentazione: non consuma la quota GPU.


In [ ]:
# 0. Internet: senza, pip e il download del dataset falliscono ("Temporary failure in name resolution")
import socket
try:
    socket.gethostbyname("pypi.org")
    print("Internet OK")
except OSError:
    raise SystemExit("Internet spento: pannello a destra -> Session options -> Internet: On "
                     "(richiede il telefono verificato in Settings), poi Run All di nuovo.")

In [ ]:
!pip install -q nibabel boto3 scipy


In [ ]:
import os, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # repository pubblico: nessun token
url = f"https://{token}@github.com/francescoluciolattari/Melampo-AI.git" if token else "https://github.com/francescoluciolattari/Melampo-AI.git"
REPO = "/kaggle/working/melampo"
if os.path.isdir(os.path.join(REPO, ".git")):
    # già scaricato in un giro precedente: si aggiorna all'ultima versione del ramo principale
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", url], check=True)
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["rm", "-rf", REPO], check=True)   # una cartella lasciata a metà da un giro interrotto
    subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Medical Segmentation Decathlon (CC BY-SA 4.0) sul registro AWS Open Data (bucket pubblico "msd-for-monai"),
# letto con boto3 senza credenziali, come indica il registro ("--no-sign-request").
# Si scaricano solo i casi che servono. Se l'elenco non trova Task06, la cella stampa cosa c'è nel bucket.
import os, glob, subprocess, sys, tarfile
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=True)
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# i casi di MSD Task06 Lung in ordine di nome: 0-30 il campione di lavoro, 30-63 i casi tenuti da parte per
# verificare una regola scelta sui primi (kaggle_api.py push holdout cambia questa riga)
CASES = slice(0, 30)
DATA = "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
images = sorted(glob.glob("/kaggle/input/**/imagesTr/lung_*.nii*", recursive=True))   # dati aggiunti con "Add Input"

for bucket, region in (("msd-for-monai", "us-west-2"), ("msd-for-monai-eu", "eu-west-2")):
    if images:
        break
    s3 = boto3.client("s3", region_name=region, config=Config(signature_version=UNSIGNED))
    try:
        keys = [o["Key"] for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket)
                for o in page.get("Contents", [])]
    except Exception as e:
        print(bucket, "elenco non disponibile:", repr(e)[:300]); continue
    print(bucket, len(keys), "oggetti; esempi:", keys[:15])
    lung = [k for k in keys if "lung" in k.lower()]
    print("con 'lung' nel nome:", lung[:20])
    imgs = sorted(k for k in lung if "imagesTr" in k and k.endswith(".nii.gz") and "/._" not in k)
    if imgs:
        for k in imgs[CASES]:
            for kk in (k, k.replace("imagesTr", "labelsTr")):
                dest = os.path.join(DATA, kk)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                if not os.path.exists(dest):
                    s3.download_file(bucket, kk, dest)
    else:
        archive = next((k for k in lung if k.endswith((".tar", ".tar.gz", ".tgz"))), None)
        if archive:
            dest = os.path.join("/kaggle/working", os.path.basename(archive))
            print("scarico l'archivio", archive, "(circa 8,5 GB)")
            s3.download_file(bucket, archive, dest)
            with tarfile.open(dest) as t:
                members = [m for m in t.getmembers() if ("imagesTr/lung_" in m.name or "labelsTr/lung_" in m.name)
                           and "/._" not in m.name]
                wanted = sorted({os.path.basename(m.name) for m in members})[CASES]
                t.extractall(DATA, members=[m for m in members if os.path.basename(m.name) in wanted])
            os.remove(dest)
    images = sorted(glob.glob(f"{DATA}/**/imagesTr/lung_*.nii.gz", recursive=True))
print(len(images), "immagini:", [os.path.basename(p) for p in images])
assert images, "nessun dato: mandare a Claude l'output di questa cella"

In [ ]:
import json, os
lobes = ["lung_upper_lobe_left", "lung_lower_lobe_left", "lung_upper_lobe_right", "lung_middle_lobe_right", "lung_lower_lobe_right"]
cases = [{"id": os.path.basename(p).split(".")[0], "image": p, "lesion": p.replace("imagesTr", "labelsTr"),
          "structure": lobes, "prompt": "tumore polmonare", "prompt_en": "lung tumor", "modality": "CT"} for p in images]
# MSD non ha referti: lo script scrive la frase di un referto con la misura di riferimento (segnata come sintetica)
json.dump(cases, open("/kaggle/working/cases.json", "w"), indent=1)
print(len(cases), "casi")

In [ ]:
# le maschere salvate dal giro con GPU (output di melampo-imaging-probe, montato in /kaggle/input)
import glob, os
_found = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/masks/lung_*.npz", recursive=True)})
print("cartelle con maschere:", _found)
if not _found:
    raise SystemExit("Nessuna maschera salvata: serve prima un giro di melampo-imaging-probe con --save-masks")
MASKS = _found[0]
print(len(glob.glob(MASKS + "/*.npz")), "casi salvati in", MASKS)
!cd /kaggle/working/melampo && python scripts/imaging_probe.py --cases /kaggle/working/cases.json --out /kaggle/working/probe.json --device cpu --anatomy --from-masks {MASKS} 2>&1 | tee /kaggle/working/probe.log


In [ ]:
import json
rows = json.load(open("/kaggle/working/probe.json"))
def mm(x):
    return "-" if x is None else f"{x:.1f}"
for r in rows:
    if "error" in r:
        print(r["id"], "ERRORE:", r["error"]); continue
    rd = r.get("reference_density") or {}
    print(f"{r['id']} ({r['slices']} fette, {r['seconds']:.0f}s) referto{' sintetico' if r['report_is_synthetic'] else ''}: "
          f"{r['report_clause']}  riferimento {mm(r['reference_longest_axial_mm'])} mm, densità rif. {rd.get('composition')} "
          f"mediana {mm(rd.get('median'))}")
    for k in ("prompt", "prompt_en"):
        if k not in r:
            continue
        v, d, s = r[k], r[k].get("density") or {}, r[k].get("second_reading") or {}
        print(f"  {k:9} {v['status']:9} Dice {v['dice']:.2f} (parte principale {v['dice_main_part']:.2f}, "
              f"soglia {mm(v.get('dice_second_reading'))})  misurato {mm(v['measured_longest_axial_mm'])} mm  "
              f"({v.get('measured_on')}, con vetro smerigliato {mm(v.get('with_ground_glass_axial_mm'))} mm, "
              f"Dice {mm(v.get('dice_with_ground_glass'))})  errore {mm(v.get('reference_error_mm'))} mm  "
              f"parti {v['parts']} altre regioni {v['other_regions']}")
        print(f"            seconda lettura {s.get('status')} accordo={s.get('agree')} soglia {mm(s.get('threshold'))} "
              f"{mm(s.get('second_axial_mm'))} mm {list(s.get('notes', []))}  supporto {v['measure_support']}")
        g = v.get("ground_glass_details") or {}
        print(f"            vetro smerigliato: soglia {mm(g.get('threshold'))} polmone {mm(g.get('surrounding'))} "
              f"rumore {mm(g.get('noise'))} voxel aggiunti {g.get('added_voxels')} {g.get('notes')}")
        print(f"            densità {d.get('composition') or d.get('relative')} mediana {mm(d.get('median'))} "
              f"centro->bordo {d.get('core_to_rim')} parte solida {mm(d.get('solid_axial_mm'))} mm {d.get('flags')}")
        print(f"            -> {'A UNA PERSONA: ' + ', '.join(v['review']) if v['needs_review'] else 'accettato'}"
              f"{'  ' + str(v['character_differences']) if v['character_differences'] else ''}")
        an, ra = v.get("anatomy") or {}, r.get("reference_anatomy") or {}
        for label, a in (("zone", an), ("zone rif.", ra)):
            if not a:
                continue
            p, g = a.get("position") or {}, a.get("margins") or {}
            inv = ", ".join(f"{i['structure']}({i['relation']} {i['lesion_share']:.2f}/{i['surround_share']:.2f})"
                            for i in a.get("involvement", [])[:5])
            print(f"            {label}: ospite {p.get('host')} {p.get('zone')} dalla superficie {mm(p.get('gap_to_surface_mm'))} mm"
                  f" | margini {g.get('contour') or '-'} {g.get('sharpness') or '-'} transizione {mm(g.get('transition_mm'))} mm"
                  f" spicole {g.get('spicules')} concavità {mm(g.get('concavity_share'))} | {inv}")
print()
print("Perché le misure sono corte (solo prompt italiano): errore del diametro rispetto al riferimento, in mm, per fascia")
variants = [("VoxTell", None), ("taglio 0.35", "cut_0.35"), ("taglio 0.2", "cut_0.2"), ("ricampionato 1.5 mm", "resampled_1_5mm"),
            ("nnInteractive", "nninteractive"), ("parti vicine unite", "merged")]
bins = [(0, 30), (30, 50), (50, 999)]
for name, key in variants:
    cells = []
    for lo, hi in bins:
        errs = []
        for r in rows:
            v = r.get("prompt") or {}
            ref, dd = r.get("reference_longest_axial_mm"), v.get("size_diagnostics") or {}
            if not ref or not (lo <= ref < hi):
                continue
            if key is None:
                m = v.get("measured_longest_axial_mm")
            elif key == "merged":
                m = dd.get("merged_near_parts_axial_mm")
            else:
                m = (dd.get(key) or {}).get("axial_mm")
            if m is not None:
                errs.append(m - ref)
        cells.append(f"{lo}-{hi if hi < 999 else '+'} mm: {sum(errs)/len(errs):+.1f} (n={len(errs)})" if errs else f"{lo}-{hi} mm: -")
    print(f"  {name:22} " + "   ".join(cells))
for r in rows:
    dd = (r.get("prompt") or {}).get("size_diagnostics") or {}
    if dd:
        print(f"  {r['id']}: volume {dd.get('volume_ratio', 0):.2f} del riferimento, mancante {dd.get('missed_share_of_reference', 0):.2f} "
              f"(entro 2 mm dal bordo {dd.get('missed_within_2mm_share', 0):.2f}, HU mediana {dd.get('missed_median_hu')}), "
              f"volume morbido {dd.get('soft_volume_ratio')}, errori: {[k for k in dd if k.endswith('error')]}")
        sh = dd.get("shape") or {}
        print(f"      forma: estensioni rif. {sh.get('extent_reference_mm')} regione {sh.get('extent_region_mm')} mm, "
              f"mancante agli estremi dell'asse {sh.get('missed_at_ends_share')}, blob mancanti {sh.get('missed_blobs')} "
              f"(il maggiore {sh.get('missed_largest_blob_share')}, HU {sh.get('missed_largest_blob_median_hu')}), "
              f"sfericità rif. {sh.get('sphericity_reference')} regione {sh.get('sphericity_region')}")
print()
for k in ("prompt", "prompt_en"):
    acc = [r[k] for r in rows if k in r and not r[k]["needs_review"]]
    rev = [r[k] for r in rows if k in r and r[k]["needs_review"]]
    ok = sum(1 for v in acc if v.get("within_strict"))
    print(f"{k}: accettati {len(acc)}, di cui entro max(2 mm, 10%) dal riferimento {ok}; a una persona {len(rev)}, "
          f"di cui davvero fuori {sum(1 for v in rev if v.get('within_strict') is False)}")
print()
print("Copia l'output di questa cella e incollalo nella chat (probe.json è aggiornato dopo ogni caso).")


# Letture da radiologo (20 ottobre): quale bordo segue il riferimento? Errore in mm rispetto al riferimento,
# sull'asse lungo e sulla media dei due assi, per fascia di dimensione
print()
print("Letture da radiologo: errore mediano (mm) asse lungo | media assi, e Dice mediano, per fascia")
import statistics as _st
_bins = [(0, 30), (30, 50), (50, 999)]
for _name in ("segmenter", "window", "window_band", "edge", "solid", "total"):
    _cells = []
    for lo, hi in _bins:
        el, em, dc = [], [], []
        for r in rows:
            rad = (r.get("prompt") or {}).get("radiologist") or {}
            ref, got = rad.get("reference") or {}, rad.get(_name) or {}
            if not ref.get("axial_long_mm") or not (lo <= ref["axial_long_mm"] < hi) or got.get("axial_long_mm") is None:
                continue
            el.append(got["axial_long_mm"] - ref["axial_long_mm"])
            if got.get("axial_mean_mm") is not None and ref.get("axial_mean_mm") is not None:
                em.append(got["axial_mean_mm"] - ref["axial_mean_mm"])
            if got.get("dice") is not None:
                dc.append(got["dice"])
        _cells.append(f"{lo}-{hi if hi < 999 else '+'} mm: n={len(el)} " + (
            f"{_st.median(el):+.1f} | {_st.median(em):+.1f}  Dice {_st.median(dc):.2f}" if el and em and dc else "-"))
    print(f"  {_name:9}", "   ".join(_cells))
for r in rows:
    rad = (r.get("prompt") or {}).get("radiologist") or {}
    if not rad or "error" in rad:
        continue
    ref = rad.get("reference") or {}
    parts = []
    for _name in ("segmenter", "window", "window_band", "edge", "solid", "total"):
        g = rad.get(_name) or {}
        if g.get("axial_long_mm") is not None:
            parts.append(f"{_name} {g['axial_long_mm']:.1f}x{(g.get('axial_short_mm') or 0):.1f} D{(g.get('dice') or 0):.2f}")
    print(f"{r['id']} rif {(ref.get('axial_long_mm') or 0):.1f}x{(ref.get('axial_short_mm') or 0):.1f} | " + " | ".join(parts)
          + f" | escluse {rad.get('excluded_voxels')} {rad.get('notes')}")
